<a href="https://colab.research.google.com/github/carlduya-tech/Prompt-Engineering/blob/main/Carl_Duya_Prompt_Engineering_Exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional


# Due input WITHOUT year. Example: 03-05 14:30
DUE_FORMAT_NO_YEAR = "%m-%d %H:%M"


@dataclass
class Task:
    title: str
    completed: bool = False
    due: Optional[datetime] = None


def print_menu() -> None:
    print("\n=== TO-DO LIST ===")
    print("1) View tasks")
    print("2) Add task")
    print("3) Delete task")
    print("4) Mark task complete")
    print("5) Quit")


def _format_due(due: Optional[datetime]) -> str:
    return "No due time" if due is None else due.strftime(DUE_FORMAT_NO_YEAR)


def _read_task_index(prompt: str, tasks: List[Task]) -> Optional[int]:
    if not tasks:
        print("No tasks to choose from.")
        return None

    raw = input(prompt).strip()
    if not raw.isdigit():
        print("Please enter a number.")
        return None

    idx = int(raw)
    if idx < 1 or idx > len(tasks):
        print(f"Please enter a number between 1 and {len(tasks)}.")
        return None

    return idx - 1  # 0-based


def _parse_due_no_year(raw: str) -> Optional[datetime]:
    """
    Parse due date/time without year: MM-DD HH:MM.
    Assumes current year; if that datetime is already in the past, rolls to next year.
    """
    try:
        parsed = datetime.strptime(raw, DUE_FORMAT_NO_YEAR)
    except ValueError:
        return None

    now = datetime.now()
    due = parsed.replace(year=now.year)
    if due < now:
        due = due.replace(year=now.year + 1)

    return due


def view_tasks(tasks: List[Task]) -> None:
    if not tasks:
        print("\nNo tasks yet.")
        return

    print("\nYour tasks:")
    for i, task in enumerate(tasks, start=1):
        status = "✓" if task.completed else " "
        print(f"{i:>2}. [{status}] {task.title}  |  Due: {_format_due(task.due)}")

    # Simple yes/no edit prompt after viewing
    ans = input("\nEdit a task? (y/n): ").strip().lower()
    if ans in ("y", "yes"):
        edit_task_flow(tasks)


def edit_task_flow(tasks: List[Task]) -> None:
    idx = _read_task_index("Enter task number to edit: ", tasks)
    if idx is None:
        return

    while True:
        task = tasks[idx]
        status = "complete" if task.completed else "not complete"
        print("\n--- EDIT TASK ---")
        print(f"Selected: {task.title}")
        print(f"Status:   {status}")
        print(f"Due:      {_format_due(task.due)}")
        print("\n1) Edit title")
        print("2) Toggle complete")
        print("3) Set due day/time")
        print("4) Clear due day/time")
        print("5) Delete task")
        print("6) Back")

        choice = input("Choose an option (1-6): ").strip()

        if choice == "1":
            new_title = input("Enter new title: ").strip()
            if not new_title:
                print("Title cannot be empty.")
            else:
                task.title = new_title
                print("Title updated.")

        elif choice == "2":
            task.completed = not task.completed
            print(f"Task is now {'complete' if task.completed else 'not complete'}.")

        elif choice == "3":
            raw = input("Enter due day/time (MM-DD HH:MM), e.g. 03-05 14:30: ").strip()
            due = _parse_due_no_year(raw)
            if due is None:
                print("Invalid format. Use MM-DD HH:MM (example: 03-05 14:30).")
            else:
                task.due = due
                print(f"Due time set -> {due.strftime('%m-%d %H:%M')}")

        elif choice == "4":
            task.due = None
            print("Due time cleared.")

        elif choice == "5":
            removed = tasks.pop(idx)
            print(f"Deleted: {removed.title}")
            return  # task no longer exists

        elif choice == "6":
            return

        else:
            print("Invalid choice. Please enter a number from 1 to 6.")


def add_task(tasks: List[Task]) -> None:
    title = input("Enter task title: ").strip()
    if not title:
        print("Task title cannot be empty.")
        return
    tasks.append(Task(title=title))
    print("Added.")


def delete_task(tasks: List[Task]) -> None:
    view_tasks(tasks)
    idx = _read_task_index("Enter task number to delete: ", tasks)
    if idx is None:
        return
    removed = tasks.pop(idx)
    print(f"Deleted: {removed.title}")


def mark_complete(tasks: List[Task]) -> None:
    view_tasks(tasks)
    idx = _read_task_index("Enter task number to mark complete: ", tasks)
    if idx is None:
        return

    if tasks[idx].completed:
        print("That task is already marked complete.")
        return

    tasks[idx].completed = True
    print(f"Completed: {tasks[idx].title}")


def main() -> None:
    tasks: List[Task] = []

    while True:
        print_menu()
        choice = input("Choose an option (1-5): ").strip()

        if choice == "1":
            view_tasks(tasks)
        elif choice == "2":
            add_task(tasks)
        elif choice == "3":
            delete_task(tasks)
        elif choice == "4":
            mark_complete(tasks)
        elif choice == "5":
            print("Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number from 1 to 5.")


if __name__ == "__main__":
    main()


=== TO-DO LIST ===
1) View tasks
2) Add task
3) Delete task
4) Mark task complete
5) Quit
Choose an option (1-5): 2
Enter task title: Haircut appointment at 3pm
Added.

=== TO-DO LIST ===
1) View tasks
2) Add task
3) Delete task
4) Mark task complete
5) Quit
Choose an option (1-5): 1

Your tasks:
 1. [ ] Haircut appointment at 3pm  |  Due: No due time

Edit a task? (y/n): y
Enter task number to edit: 1

--- EDIT TASK ---
Selected: Haircut appointment at 3pm
Status:   not complete
Due:      No due time

1) Edit title
2) Toggle complete
3) Set due day/time
4) Clear due day/time
5) Delete task
6) Back
Choose an option (1-6): 3
Enter due day/time (MM-DD HH:MM), e.g. 03-05 14:30: 02-28 14:00
Due time set -> 02-28 14:00

--- EDIT TASK ---
Selected: Haircut appointment at 3pm
Status:   not complete
Due:      02-28 14:00

1) Edit title
2) Toggle complete
3) Set due day/time
4) Clear due day/time
5) Delete task
6) Back
Choose an option (1-6): 6

=== TO-DO LIST ===
1) View tasks
2) Add task
3) 

KeyboardInterrupt: Interrupted by user